# AR Step 3 MLP Probes For Missing Properties

Runs the same ridge-vs-MLP raw/residual probe study as `ar_step3_mlp_probes.ipynb`, but only for the properties present in `ProbeVAE/latent-analisys.ipynb` and absent from the current AR Step 3 property panel: `BertzCT` and `QED`.

In [ ]:
from pathlib import Path
import copy
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

def find_project_root(start=None):
    p = (Path(start) if start is not None else Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    raise RuntimeError(f"Couldn't find project root from {p}")

# Same canonical project root used by ProbeVAE/latent-analisys.ipynb, with a
# local fallback so the notebook also runs from a checked-out repository.
LATENT_ANALYSIS_PROJECT_ROOT = Path(".")
try:
    PROJECT_ROOT = find_project_root(start=LATENT_ANALYSIS_PROJECT_ROOT) if LATENT_ANALYSIS_PROJECT_ROOT.exists() else find_project_root()
except RuntimeError:
    PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from artifacts.model_compare.ar_model_h256_l256.patched_notebooks import ar_common as arc

ARTIFACT_ROOT = arc.ARTIFACT_ROOT
LATENT_ANALYSIS_PANEL_DIR = PROJECT_ROOT / "artifacts" / "latent_analysis_panels"
PROPERTIES_CSV = LATENT_ANALYSIS_PANEL_DIR / "molecule_properties_panel.csv"
CONFOUNDS_CSV = LATENT_ANALYSIS_PANEL_DIR / "confound_features_panel.csv"
LATENTS_NPY = LATENT_ANALYSIS_PANEL_DIR / "latents_mu.npy"
LATENTS_NPZ = LATENT_ANALYSIS_PANEL_DIR / "latents_mu_with_meta.npz"

print("project_root =", PROJECT_ROOT)
print("artifact_root =", ARTIFACT_ROOT)
print("data_csv =", arc.DATASET_PATH)
print("checkpoint =", arc.checkpoint_path())
print("properties_csv =", PROPERTIES_CSV)
print("confounds_csv =", CONFOUNDS_CSV)
print("latents_npy =", LATENTS_NPY)
print("latents_npz =", LATENTS_NPZ)
for warning_text in arc.dependency_report():
    warnings.warn(warning_text)

In [ ]:
OUTPUT_DIR = ARTIFACT_ROOT / "confounds_mlp_step3_missing_properties"
TABLES_DIR = OUTPUT_DIR / "tables"
PLOTS_DIR = OUTPUT_DIR / "plots"
for path in [OUTPUT_DIR, TABLES_DIR, PLOTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)


## Missing Property Targets

In [ ]:
LATENT_ANALYSIS_STUDIED_PROPERTIES = [
    "molWt", "HeavyAtomCount", "cLogP", "TPSA", "HBD", "HBA",
    "NumRotatableBonds", "RingCount", "AromaticRingCount",
    "FractionCSP3", "NumSpiroAtoms", "NumBridgeheadAtoms", "BertzCT", "QED",
]

TARGET_PROPERTIES = [
    prop for prop in LATENT_ANALYSIS_STUDIED_PROPERTIES
    if prop not in set(arc.AR_PROPERTY_COLUMNS)
]
print("latent-analysis properties missing from AR_PROPERTY_COLUMNS =", TARGET_PROPERTIES)

## Load Latent-Analysis Panels And Latents

In [ ]:
required_inputs = {
    "properties_csv": PROPERTIES_CSV,
    "confounds_csv": CONFOUNDS_CSV,
    "latents_npy": LATENTS_NPY,
}
missing = {name: path for name, path in required_inputs.items() if not path.exists()}
if missing:
    lines = [
        "Missing latent-analysis cached inputs. Run ProbeVAE/latent-analisys.ipynb cells that build the panels and save latents.",
        "Expected paths:",
    ]
    lines.extend(f"  {name}: {path}" for name, path in required_inputs.items())
    raise FileNotFoundError("\n".join(lines))

Y = pd.read_csv(PROPERTIES_CSV)
X = pd.read_csv(CONFOUNDS_CSV)
Z = np.load(LATENTS_NPY).astype(np.float32)

n = len(Z)
if len(Y) != n or len(X) != n:
    raise ValueError(f"latent-analysis row mismatch: len(Y)={len(Y)}, len(X)={len(X)}, len(Z)={n}")

all_idx = np.arange(n)
train_idx, temp_idx = arc.train_test_split(all_idx, test_size=0.2, random_state=arc.SEED, shuffle=True)
val_idx, test_idx = arc.train_test_split(temp_idx, test_size=0.5, random_state=arc.SEED, shuffle=True)
split = np.full(n, "", dtype=object)
split[train_idx] = "train"
split[val_idx] = "val"
split[test_idx] = "test"

meta_df = pd.DataFrame({"split": split})
if LATENTS_NPZ.exists():
    with np.load(LATENTS_NPZ, allow_pickle=True) as zmeta:
        for key in ["smiles", "selfies"]:
            if key in zmeta and len(zmeta[key]) == n:
                meta_df[key] = zmeta[key]

panel = pd.concat(
    [meta_df.reset_index(drop=True), X.reset_index(drop=True), Y.reset_index(drop=True)],
    axis=1,
)
panel.to_csv(OUTPUT_DIR / "latent_analysis_panel_for_missing_mlp.csv", index=False)

C_mat = panel[arc.CONF_COLUMNS].to_numpy(dtype=float)
print("target properties =", TARGET_PROPERTIES)
print("latent shape =", Z.shape)
print("properties source =", PROPERTIES_CSV)
print("confounds source =", CONFOUNDS_CSV)
print("latents source =", LATENTS_NPY)
display(panel.head())

## Add BertzCT And QED To The Panel

In [ ]:
needs_compute = [prop for prop in TARGET_PROPERTIES if prop not in panel.columns]
if needs_compute:
    if "smiles" not in panel.columns:
        raise KeyError(
            "The latent-analysis property cache is missing "
            f"{needs_compute}, and no smiles metadata was found for fallback RDKit computation. "
            f"Expected these columns in {PROPERTIES_CSV}."
        )
    arc.require_dependencies("rdkit")
    from rdkit import Chem
    from rdkit.Chem import Descriptors, QED

    def compute_missing_properties(smiles_values):
        rows = []
        for smiles in tqdm(smiles_values, desc="missing RDKit properties"):
            mol = Chem.MolFromSmiles(str(smiles)) if pd.notna(smiles) else None
            if mol is None:
                rows.append({"BertzCT": np.nan, "QED": np.nan})
            else:
                rows.append({
                    "BertzCT": float(Descriptors.BertzCT(mol)),
                    "QED": float(QED.qed(mol)),
                })
        return pd.DataFrame(rows)

    missing_df = compute_missing_properties(panel["smiles"])
    for prop in needs_compute:
        panel[prop] = missing_df[prop].to_numpy()

target_properties = [
    prop for prop in TARGET_PROPERTIES
    if prop in panel.columns and panel[prop].notna().sum() > 100
]
if not target_properties:
    raise RuntimeError("No missing properties had enough valid values for probe training.")

preview_cols = [col for col in ["smiles", "selfies", *target_properties] if col in panel.columns]
print("usable missing properties =", target_properties)
display(panel[preview_cols].head())

## MLP Helper Functions

In [ ]:
torch = arc.torch
nn = torch.nn
DataLoader = torch.utils.data.DataLoader
TensorDataset = torch.utils.data.TensorDataset

BATCH_SIZE = 8192
MAX_EPOCHS = 15
PATIENCE = 4
LR = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_WIDTH = 256
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class MLPProbe(nn.Module):
    def __init__(self, in_dim, hidden_width=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_width),
            nn.GELU(),
            nn.LayerNorm(hidden_width),
            nn.Linear(hidden_width, hidden_width),
            nn.GELU(),
            nn.Linear(hidden_width, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

def fit_mlp_probe(X, y, split):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    split = np.asarray(split)
    finite = np.isfinite(y) & np.isfinite(X).all(axis=1)
    masks = {name: (split == name) & finite for name in ["train", "val", "test"]}

    x_scaler = arc.StandardScaler().fit(X[masks["train"]])
    y_mean = float(y[masks["train"]].mean())
    y_std = float(y[masks["train"]].std() + 1e-8)
    Xs = x_scaler.transform(X).astype(np.float32)
    ys = ((y - y_mean) / y_std).astype(np.float32)

    def loader_for(mask, shuffle=False):
        ds = TensorDataset(torch.from_numpy(Xs[mask]), torch.from_numpy(ys[mask]))
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

    model = MLPProbe(X.shape[1], HIDDEN_WIDTH).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.MSELoss()
    best_state = None
    best_val = np.inf
    stale = 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        for xb, yb in loader_for(masks["train"], shuffle=True):
            xb = xb.to(device)
            yb = yb.to(device)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in loader_for(masks["val"], shuffle=False):
                xb = xb.to(device)
                yb = yb.to(device)
                val_losses.append(float(loss_fn(model(xb), yb).detach().cpu()))
        val_loss = float(np.mean(val_losses))
        if val_loss < best_val - 1e-5:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        if stale >= PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    preds = {}
    model.eval()
    with torch.no_grad():
        for name in ["train", "val", "test"]:
            out = []
            for xb, _ in loader_for(masks[name], shuffle=False):
                pred = model(xb.to(device)).detach().cpu().numpy()
                out.append(pred)
            preds[name] = np.concatenate(out) * y_std + y_mean

    return {
        "r2_train": arc.safe_r2(y[masks["train"]], preds["train"]),
        "r2_val": arc.safe_r2(y[masks["val"]], preds["val"]),
        "r2_test": arc.safe_r2(y[masks["test"]], preds["test"]),
        "epochs": int(epoch + 1),
    }

## Fit Ridge And MLP On Raw And Residual Targets

In [ ]:
ridge_raw_rows = []
ridge_resid_rows = []
c_to_y_rows = []
mlp_raw_rows = []
mlp_resid_rows = []

for prop in tqdm(target_properties, desc="missing-property probes"):
    y = panel[prop].to_numpy(dtype=float)

    _, _, _, _, z_scores = arc.fit_ridge_probe(Z, y, split, alpha=1e-2)
    ridge_raw_rows.append({"property": prop, **z_scores})

    residual, c_model, c_scaler, c_scores = arc.residualize_against_confounds(C_mat, y, split, alpha=1e1)
    resid_col = f"resid_{prop}"
    panel[resid_col] = residual
    c_to_y_rows.append({"property": prop, **c_scores})

    _, _, _, _, zr_scores = arc.fit_ridge_probe(Z, residual, split, alpha=1e-2)
    ridge_resid_rows.append({
        "property": prop,
        "target": resid_col,
        "r2_original": z_scores["r2_val"],
        "r2_residual": zr_scores["r2_val"],
        "delta": zr_scores["r2_val"] - z_scores["r2_val"],
        "r2_original_test": z_scores["r2_test"],
        "r2_residual_test": zr_scores["r2_test"],
        "delta_test": zr_scores["r2_test"] - z_scores["r2_test"],
        "alpha_original": z_scores["alpha"],
        "alpha_residual": zr_scores["alpha"],
    })

    mlp_raw_scores = fit_mlp_probe(Z, y, split)
    mlp_raw_rows.append({"property": prop, **mlp_raw_scores})

    mlp_resid_scores = fit_mlp_probe(Z, residual, split)
    mlp_resid_rows.append({"property": prop, "target": resid_col, **mlp_resid_scores})

ridge_raw = pd.DataFrame(ridge_raw_rows)
ridge_resid = pd.DataFrame(ridge_resid_rows)
r2_C_to_Y = pd.DataFrame(c_to_y_rows)
mlp_raw = pd.DataFrame(mlp_raw_rows)
mlp_resid = pd.DataFrame(mlp_resid_rows)

ridge_raw.to_csv(TABLES_DIR / "r2_Z_to_Y_missing_ridge.csv", index=False)
ridge_resid.to_csv(TABLES_DIR / "r2_Z_to_Yresid_missing_ridge.csv", index=False)
r2_C_to_Y.to_csv(TABLES_DIR / "r2_C_to_Y_missing.csv", index=False)
mlp_raw.to_csv(TABLES_DIR / "r2_Z_to_Y_missing_mlp.csv", index=False)
mlp_resid.to_csv(TABLES_DIR / "r2_Z_to_Yresid_missing_mlp.csv", index=False)
panel.to_csv(OUTPUT_DIR / "panels_step3_missing_properties_with_residuals.csv", index=False)

display(mlp_raw)
display(mlp_resid)

## Compare MLP To Ridge

In [ ]:
raw_compare = mlp_raw.merge(
    ridge_raw[["property", "r2_test"]].rename(columns={"r2_test": "ridge_raw_r2_test"}),
    on="property",
    how="left",
)
raw_compare["mlp_raw_r2_test"] = raw_compare["r2_test"]
raw_compare["Delta-R2_raw"] = raw_compare["mlp_raw_r2_test"] - raw_compare["ridge_raw_r2_test"]
raw_compare.to_csv(TABLES_DIR / "mlp_vs_ridge_missing_raw.csv", index=False)

resid_compare = mlp_resid.merge(
    ridge_resid[["property", "r2_residual_test"]].rename(columns={"r2_residual_test": "ridge_residual_r2_test"}),
    on="property",
    how="left",
)
resid_compare["mlp_residual_r2_test"] = resid_compare["r2_test"]
resid_compare["Delta-R2_residual"] = resid_compare["mlp_residual_r2_test"] - resid_compare["ridge_residual_r2_test"]
resid_compare.to_csv(TABLES_DIR / "mlp_vs_ridge_missing_resid.csv", index=False)

overview = raw_compare[["property", "ridge_raw_r2_test", "mlp_raw_r2_test", "Delta-R2_raw"]].merge(
    resid_compare[["property", "ridge_residual_r2_test", "mlp_residual_r2_test", "Delta-R2_residual"]],
    on="property",
    how="outer",
)

corr_long = arc.property_confound_correlations(panel, target_properties, arc.CONF_COLUMNS)
corr_long.to_csv(TABLES_DIR / "property_confound_correlations_missing.csv", index=False)
if len(corr_long):
    max_corr = corr_long.assign(abs_spearman=lambda d: d["spearman_corr"].abs()).groupby("property", as_index=False)["abs_spearman"].max()
    overview = overview.merge(max_corr, on="property", how="left")
else:
    overview["abs_spearman"] = np.nan

def classify(row):
    if row.get("abs_spearman", 0) >= 0.80:
        return "confounded property"
    if row.get("ridge_residual_r2_test", 0) >= 0.50 and row.get("Delta-R2_residual", 0) < 0.10:
        return "globally linear property"
    if row.get("Delta-R2_residual", 0) >= 0.15:
        return "nonlinear/local property"
    if row.get("mlp_residual_r2_test", 0) < 0.20:
        return "weakly represented property"
    return "mixed property"

overview["interpretation"] = overview.apply(classify, axis=1)
overview.to_csv(TABLES_DIR / "mlp_ridge_missing_delta_r2_overview.csv", index=False)
display(overview.sort_values("Delta-R2_residual", ascending=False))

## Minimal Plots And Summary

In [ ]:
def barh_metric(df, value, title, filename):
    plot_df = df.sort_values(value, ascending=True)
    plt.figure(figsize=(8, max(3, 0.8 * len(plot_df))))
    plt.barh(plot_df["property"], plot_df[value])
    plt.axvline(0, color="black", linewidth=1)
    plt.xlabel(value)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / filename, dpi=180)
    plt.show()

barh_metric(raw_compare, "Delta-R2_raw", "Delta-R2 raw: MLP minus ridge", "delta_r2_missing_raw.png")
barh_metric(resid_compare, "Delta-R2_residual", "Delta-R2 residual: MLP minus ridge", "delta_r2_missing_residual.png")

plt.figure(figsize=(5, 5))
plt.scatter(raw_compare["ridge_raw_r2_test"], raw_compare["mlp_raw_r2_test"])
lims = [0, 1]
plt.plot(lims, lims, color="black", linewidth=1)
plt.xlim(lims)
plt.ylim(lims)
plt.xlabel("ridge raw R^2")
plt.ylabel("MLP raw R^2")
plt.title("Ridge vs MLP raw R^2")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "ridge_vs_mlp_missing_raw_r2.png", dpi=180)
plt.show()

plt.figure(figsize=(5, 5))
plt.scatter(resid_compare["ridge_residual_r2_test"], resid_compare["mlp_residual_r2_test"])
plt.plot(lims, lims, color="black", linewidth=1)
plt.xlim(lims)
plt.ylim(lims)
plt.xlabel("ridge residual R^2")
plt.ylabel("MLP residual R^2")
plt.title("Ridge vs MLP residual R^2")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "ridge_vs_mlp_missing_residual_r2.png", dpi=180)
plt.show()

summary = {
    "artifact_root": str(OUTPUT_DIR),
    "source_latent_analysis_notebook": "ProbeVAE/latent-analisys.ipynb",
    "data_csv": str(arc.DATASET_PATH),
    "checkpoint": str(arc.checkpoint_path()),
    "properties_csv": str(PROPERTIES_CSV),
    "confounds_csv": str(CONFOUNDS_CSV),
    "latents_npy": str(LATENTS_NPY),
    "latents_npz": str(LATENTS_NPZ),
    "n_properties": int(len(target_properties)),
    "properties": target_properties,
    "avg_raw_Delta-R2": float(raw_compare["Delta-R2_raw"].mean()),
    "avg_residual_Delta-R2": float(resid_compare["Delta-R2_residual"].mean()),
}
arc.write_json(OUTPUT_DIR / "summary_mlp_missing_properties.json", summary)
display(pd.DataFrame([summary]))